# Classificação de comentários sobre o jogo Aviator com Llama 3.1 8B Instruct e QLoRA

**Trabalho de Conclusão de Curso — Sistemas de Informação**  
**Autor:** Velber Gonçalves Fernandes  
**Instituição:** Instituto Federal de Minas Gerais (IFMG) — Campus Sabará

---

## 1. Apresentação

Este notebook documenta o procedimento experimental de ajuste supervisionado do modelo **Llama 3.1 8B Instruct**, por meio da técnica **QLoRA (Quantized Low-Rank Adaptation)**, para classificar comentários do YouTube relacionados ao jogo Aviator.

A tarefa consiste em identificar o posicionamento expresso em cada comentário, considerando três categorias:

- **Favorável:** demonstra apoio, aprovação ou avaliação positiva;
- **Contrário:** demonstra crítica, rejeição ou avaliação negativa;
- **Neutro:** não apresenta posicionamento claramente favorável ou contrário.

A coluna `rotulo_final` contém a classificação de referência previamente atribuída no processo de anotação do estudo.

## 2. Objetivo e escopo da execução

O objetivo é ajustar o modelo com comentários rotulados e avaliar sua capacidade de classificar comentários que não participaram do treinamento daquela execução.

A base utilizada contém **1.100 comentários**, distribuídos previamente em cinco partições (*folds*). Em cada execução, uma partição é reservada para validação e as demais são utilizadas no treinamento.

<escape>**Atenção metodológica:**</escape> o código desta versão executa os cinco folds (IDs `0`, `1`, `2`, `3` e `4`). Em cada execução, um fold é reservado para validação e os outros quatro são utilizados no treinamento. Ao final, cada um dos 1.100 comentários terá sido avaliado uma vez, desde que a base e o arquivo de folds contenham todos os registros esperados.

## 3. Arquivos de entrada

O notebook espera os seguintes arquivos CSV no diretório `/content/` do Google Colab:

| Arquivo | Conteúdo |
|---|---|
| `base_tcc_final_1100.csv` | Comentários, identificadores e rótulos de referência |
| `folds_1100.csv` | Identificador do comentário e respectivo fold |

A associação entre os arquivos é realizada pela coluna `youtube_comment_id`.

Colunas principais:
- `youtube_comment_id`: identificador único do comentário;
- `comment_text`: texto submetido ao modelo;
- `rotulo_final`: classe de referência;
- `fold`: partição utilizada na avaliação.

As colunas `youtube_video_id` e `period`, quando existentes, são preservadas nos arquivos de predição.

## 4. Procedimento de treinamento

O modelo-base é carregado novamente em cada fold. A quantização em 4 bits reduz o consumo de memória, enquanto os adaptadores LoRA permitem ajustar um conjunto reduzido de parâmetros.

Cada comentário é inserido em um prompt padronizado que solicita uma resposta correspondente a uma das três classes. Durante o treinamento, os tokens do prompt são mascarados para que a função de perda seja calculada sobre os tokens da resposta esperada.

Na inferência, o modelo gera uma resposta e o código tenta identificar uma única classe válida. Respostas ambíguas ou que não correspondam a uma das classes são registradas como `Inválido`.

## 5. Configuração experimental

| Parâmetro | Valor configurado |
|---|---:|
| Modelo-base | `meta-llama/Llama-3.1-8B-Instruct` |
| Quantização | 4 bits, NF4 |
| Dupla quantização | Ativada |
| Rank LoRA (`r`) | 8 |
| `lora_alpha` | 16 |
| `lora_dropout` | 0,05 |
| Comprimento máximo | 256 tokens |
| Épocas por fold | 1 |
| Lote por dispositivo | 1 |
| Acumulação de gradientes | 8 |
| Taxa de aprendizado | 0,0002 |
| Semente aleatória | 42 |

Os valores apresentados correspondem ao código deste notebook. Caso sejam alterados nos experimentos, a configuração efetivamente utilizada deve ser registrada junto aos resultados.

## 6. Métricas de avaliação

São calculadas:
- **Acurácia:** proporção de previsões corretas;
- **Precisão macro:** média não ponderada da precisão das três classes;
- **F1-score macro:** média não ponderada dos F1-scores das três classes;
- **F1-score por classe:** desempenho individual em Favorável, Contrário e Neutro;
- **Matriz de confusão:** comparação entre classes reais e previstas.

As respostas inválidas não são removidas das métricas gerais: elas são consideradas incorretas na acurácia e prejudicam a recuperação da classe verdadeira no F1-score. Na matriz de confusão produzida pelo código original, entretanto, as respostas inválidas não aparecem em uma coluna própria; por isso, a matriz deve ser interpretada em conjunto com a contagem de respostas inválidas.

## 7. Ordem de execução

1. Instalar as dependências;
2. Autenticar no Hugging Face;
3. Verificar o ambiente computacional e carregar os dados;
4. Configurar o tokenizer e o modelo QLoRA;
5. Executar o treinamento e as previsões dos folds selecionados;
6. Consolidar as métricas e salvar os resultados;
7. Compactar e baixar os arquivos gerados.

Execute as células na ordem apresentada. O treinamento requer GPU compatível com as bibliotecas utilizadas.

## Etapa 1 — Instalação das dependências

Instala as bibliotecas utilizadas para carregar o modelo, realizar o ajuste QLoRA, processar os dados e calcular as métricas.

No Google Colab, reinicie o ambiente de execução caso a plataforma solicite.

In [ ]:
%pip install -q -U transformers peft accelerate bitsandbytes datasets scikit-learn pandas tqdm huggingface_hub

## Etapa 2 — Autenticação no Hugging Face

O repositório do modelo Meta Llama pode exigir autenticação e aceite prévio dos termos de acesso. Execute a célula e informe o token pelo mecanismo seguro disponibilizado pela biblioteca.

**Não inclua tokens ou credenciais no código nem compartilhe um notebook que contenha informações de autenticação.**

In [ ]:
from huggingface_hub import login
login()

## Etapa 3 — Verificação do ambiente e preparação dos dados

Nesta etapa são importadas as bibliotecas, verificadas a disponibilidade da GPU e a presença dos arquivos de entrada. Em seguida, a base de comentários é associada ao arquivo de folds pela chave `youtube_comment_id`.

O código também verifica a unicidade dos identificadores, a ausência de textos e rótulos vazios, a presença das três classes esperadas e a existência dos cinco folds.

In [ ]:
import os
# Configura o PyTorch para usar segmentos expansíveis e evitar fragmentação de memória de GPU
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json, gc, inspect, time
import numpy as np
import pandas as pd
import torch
import transformers, peft, bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
assert torch.cuda.is_available(), "GPU não disponível. No Colab, selecione Ambiente de execução > Alterar tipo de ambiente de execução > GPU."
print("GPU:", torch.cuda.get_device_name(0))

BASE_PATH = "/content/base_tcc_final_1100.csv"
FOLDS_PATH = "/content/folds_1100.csv"
assert os.path.exists(BASE_PATH), f"Arquivo não encontrado: {BASE_PATH}"
assert os.path.exists(FOLDS_PATH), f"Arquivo não encontrado: {FOLDS_PATH}"

base = pd.read_csv(BASE_PATH)
folds = pd.read_csv(FOLDS_PATH)
print("Colunas da base:", list(base.columns))
print("Colunas dos folds:", list(folds.columns))

# Validar esquema e unir base + folds pela chave persistente do comentário
required_base = {"youtube_comment_id", "comment_text", "rotulo_final"}
required_folds = {"youtube_comment_id", "fold"}
assert required_base.issubset(base.columns), f"Faltam colunas na base: {required_base - set(base.columns)}"
assert required_folds.issubset(folds.columns), f"Faltam colunas nos folds: {required_folds - set(folds.columns)}"
assert not base["youtube_comment_id"].duplicated().any(), "Há IDs duplicados na base."
assert not folds["youtube_comment_id"].duplicated().any(), "Há IDs duplicados nos folds."
assert base["comment_text"].notna().all(), "Há comentários sem texto."
assert base["rotulo_final"].notna().all(), "Há rótulos ausentes."

dados = base.merge(
    folds[["youtube_comment_id", "fold"]],
    on="youtube_comment_id",
    how="left",
    validate="one_to_one"
)
assert dados["fold"].notna().all(), "Há comentários sem fold."
dados["fold"] = dados["fold"].astype(int)
CLASSES = ["Favorável", "Contrário", "Neutro"]
assert set(dados["rotulo_final"].unique()) == set(CLASSES), f"Rótulos inesperados: {set(dados['rotulo_final'].unique())}"
assert len(dados) == 1100, f"Esperados 1100 comentários; encontrados {len(dados)}."
assert sorted(dados["fold"].unique().tolist()) == [0,1,2,3,4], "Esperados folds 0,1,2,3,4."

print("Total:", len(dados))
print("\nDistribuição dos rótulos:")
print(dados["rotulo_final"].value_counts())
print("\nDistribuição por fold:")
print(pd.crosstab(dados["fold"], dados["rotulo_final"]))
print("\nValidação de integridade concluída.")

## Etapa 4 — Configuração do modelo e do treinamento

O modelo é carregado com quantização de 4 bits (NF4), preparado para treinamento com baixa quantidade de bits e adaptado com LoRA nos módulos de projeção de atenção definidos em `target_modules`.

A função `criar_prompt` padroniza a instrução de classificação. A função `tokenizar_exemplo` concatena o prompt à resposta esperada e mascara os tokens do prompt com `-100`, de modo que a perda seja calculada sobre a resposta.

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    Trainer, TrainingArguments
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import torch
import inspect
import os

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
SEED = 42
MAX_LENGTH = 256
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
OUTPUT_DIR = "/content/resultados_llama_1100"
os.makedirs(OUTPUT_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Correção Crítica Llama 3: usar token de padding dedicado para evitar conflito com EOS
tokenizer.pad_token = "<|finetune_right_pad_id|>"
tokenizer.padding_side = "right"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

def criar_prompt(comentario):
    return (
        "Classifique o posicionamento do comentário sobre o jogo Aviator "
        "em uma das seguintes classes: Favorável, Contrário ou Neutro.\n"
        "Responda somente com o nome da classe.\n\n"
        f"Comentário: {comentario}\n"
        "Classificação:"
    )

def tokenizar_exemplo(exemplo):
    prompt = criar_prompt(str(exemplo["comment_text"]))
    resposta = " " + str(exemplo["rotulo_final"]) + "<|eot_id|>"
    tokens_prompt = tokenizer(
        prompt, add_special_tokens=True, truncation=True,
        max_length=MAX_LENGTH - 10
    )
    tokens_resposta = tokenizer(resposta, add_special_tokens=False)
    input_ids = tokens_prompt["input_ids"] + tokens_resposta["input_ids"]
    labels = [-100] * len(tokens_prompt["input_ids"]) + tokens_resposta["input_ids"]
    input_ids = input_ids[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels
    }

class CollatorCausalLM:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, examples):
        max_len = max(len(x["input_ids"]) for x in examples)
        input_ids, attention_mask, labels = [], [], []
        for x in examples:
            pad = max_len - len(x["input_ids"])
            input_ids.append(x["input_ids"] + [self.tokenizer.convert_tokens_to_ids("<|finetune_right_pad_id|>")] * pad)
            attention_mask.append(x["attention_mask"] + [0] * pad)
            labels.append(x["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

def carregar_modelo_qlora():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    )
    model = get_peft_model(model, lora_config)
    return model

def criar_training_args(fold, output_dir):
    params = inspect.signature(TrainingArguments.__init__).parameters
    kwargs = dict(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=0,
        fp16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        save_strategy="no",
        logging_steps=10,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        dataloader_pin_memory=True
    )
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "no"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "no"
    return TrainingArguments(**kwargs)

import re
import unicodedata

def normalizar_texto(texto):
    texto = unicodedata.normalize("NFKC", str(texto)).casefold()
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def extrair_classe(resposta):
    texto = normalizar_texto(resposta)
    encontradas = []
    for classe in CLASSES:
        classe_norm = normalizar_texto(classe)
        padrao = r"(?<!\w)" + re.escape(classe_norm) + r"(?!\w)"
        if re.search(padrao, texto):
            encontradas.append(classe)
    if len(encontradas) == 1:
        return encontradas[0]
    return "Inválido"

print("Configuração limpa e corrigida para Llama 3.")

## Etapa 5 — Treinamento e geração das previsões

O código executa os cinco folds, de IDs `0` a `4`. Em cada fold, os comentários daquela partição são reservados para validação e os demais quatro folds são utilizados no treinamento.

Um novo modelo é inicializado em cada execução. Após o treinamento, são geradas previsões para os comentários de validação, preservando o texto original da resposta para permitir auditoria posterior.

**Importante:** a execução completa exige que todos os cinco folds terminem com sucesso. Se houver interrupção, confira os arquivos individuais de cada fold antes de interpretar os resultados consolidados.

In [ ]:
print("Folds disponíveis:", sorted(dados["fold"].unique()))

for fold_id in sorted(dados["fold"].unique()):
    quantidade = (dados["fold"] == fold_id).sum()
    print(f"Fold ID {fold_id}: {quantidade} comentários")

import os

OUTPUT_DIR = "/content/resultados_llama_1100"

print("=== Arquivos de predições ===")

for fold_id in range(5):
    arquivo = f"predicoes_fold_{fold_id}.csv"
    caminho = os.path.join(OUTPUT_DIR, arquivo)

    if os.path.exists(caminho):
        print(f"Fold {fold_id}: encontrado")
    else:
        print(f"Fold {fold_id}: não encontrado")

print("\n=== Checkpoints encontrados ===")

for raiz, pastas, arquivos in os.walk(OUTPUT_DIR):
    for pasta in pastas:
        if "checkpoint" in pasta.lower():
            print(os.path.join(raiz, pasta))

# Teste local: não carrega o modelo nem utiliza a GPU
testes = [
    ("Contrário", "Contrário"),
    ("Neutro", "Neutro"),
    ("Favorável", "Favorável"),
    ("A classificação é: Contrário.", "Contrário"),
    ("Resposta: Favorável", "Favorável"),
]

for resposta, esperado in testes:
    obtido = extrair_classe(resposta)
    status = "OK" if obtido == esperado else "ERRO"
    print(f"{status} | Entrada: {resposta!r} | Obtido: {obtido}")

def prever_classe_corrigido(texto, model):
    prompt = criar_prompt(str(texto))

    # Garante padding à esquerda na inferência
    tokenizer.padding_side = "left"
    entradas = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    tokenizer.padding_side = "right"

    try:
        device = model.get_input_embeddings().weight.device
    except Exception:
        device = "cuda"

    entradas = {k: v.to(device) for k, v in entradas.items()}

    # Terminators específicos do Llama 3
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        tokenizer.convert_tokens_to_ids("<|finetune_right_pad_id|>")
    ]

    with torch.no_grad():
        saida = model.generate(
            **entradas,
            max_new_tokens=8,
            do_sample=False, # Geração gulosa (greedy) sem alucinações
            pad_token_id=tokenizer.convert_tokens_to_ids("<|finetune_right_pad_id|>"),
            eos_token_id=terminators
        )

    novos = saida[0][entradas["input_ids"].shape[1]:]
    resposta_bruta = tokenizer.decode(novos, skip_special_tokens=True).strip()
    classe = extrair_classe(resposta_bruta)
    return classe, resposta_bruta

predicoes_por_fold = []
metricas_por_fold = []

# Validação cruzada completa: cinco folds, IDs 0, 1, 2, 3 e 4.
folds_alvo = [0, 1, 2, 3, 4]

folds_disponiveis = sorted(dados["fold"].unique().tolist())
if folds_disponiveis != folds_alvo:
    raise ValueError(
        "A validação cruzada completa exige os folds 0, 1, 2, 3 e 4. "
        f"Folds encontrados: {folds_disponiveis}"
    )

if len(dados) != 1100:
    raise ValueError(
        f"Esta execução espera 1.100 comentários; foram encontrados {len(dados)}."
    )

for fold_id in folds_alvo:
    print("\n" + "=" * 60)
    print(f"EXECUTANDO FOLD {fold_id + 1}/5")
    print("=" * 60)

    treino_df = dados[dados["fold"] != fold_id].copy().reset_index(drop=True)
    validacao_df = dados[dados["fold"] == fold_id].copy().reset_index(drop=True)
    print("Treino:", len(treino_df), "| Validação:", len(validacao_df))

    treino_ds = Dataset.from_pandas(treino_df[["comment_text", "rotulo_final"]])
    treino_tok = treino_ds.map(tokenizar_exemplo, remove_columns=treino_ds.column_names)

    model = carregar_modelo_qlora()
    model.print_trainable_parameters()
    args = criar_training_args(fold_id, f"{OUTPUT_DIR}/fold_{fold_id}")
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=treino_tok,
        data_collator=CollatorCausalLM(tokenizer)
    )
    treino_resultado = trainer.train()

    model.eval()
    previsoes = []
    respostas_brutas = []

    for texto in tqdm(validacao_df["comment_text"].tolist(), desc=f"Predições fold {fold_id + 1}"):
        classe_prevista, resposta_bruta = prever_classe_corrigido(texto, model)
        previsoes.append(classe_prevista)
        respostas_brutas.append(resposta_bruta)

    saida_fold = validacao_df[[
        "youtube_comment_id", "youtube_video_id", "period",
        "comment_text", "rotulo_final", "fold"
    ]].copy()
    saida_fold["predicao_llama"] = previsoes
    saida_fold["resposta_bruta_llama"] = respostas_brutas
    saida_fold.to_csv(f"{OUTPUT_DIR}/predicoes_fold_{fold_id}.csv", index=False, encoding="utf-8-sig")
    predicoes_por_fold.append(saida_fold)

    # Avaliação das métricas
    validos = saida_fold["predicao_llama"].isin(CLASSES)
    y_true = saida_fold["rotulo_final"]
    y_pred = saida_fold["predicao_llama"]
    met = {
        "fold": int(fold_id),
        "n_validacao": int(len(saida_fold)),
        "n_validas": int(validos.sum()),
        "n_invalidas": int((~validos).sum()),
        "macro_f1": float(f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0)),
        "precisao_macro": float(precision_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0)),
        "acuracia": float(accuracy_score(y_true, y_pred)),
    }
    for classe_nome in CLASSES:
        met[f"f1_{classe_nome}"] = float(f1_score(y_true, y_pred, labels=[classe_nome], average="macro", zero_division=0))
    metricas_por_fold.append(met)
    pd.DataFrame(metricas_por_fold).to_csv(f"{OUTPUT_DIR}/metricas_por_fold_parcial.csv", index=False)

    print("Métricas do fold:", met)

    del trainer, model, treino_ds, treino_tok, treino_df, validacao_df
    gc.collect()
    torch.cuda.empty_cache()

print("\nValidação cruzada de cinco folds concluída.")

## Etapa 6 — Consolidação das métricas e salvamento

As previsões e métricas dos folds executados são reunidas em arquivos CSV e JSON. As métricas consolidadas consideram os cinco folds executados.

A matriz de confusão utiliza as três classes de referência. Como as respostas inválidas não fazem parte dessas classes, elas não aparecem como coluna na matriz; consulte também a contagem de previsões inválidas apresentada no resumo.

In [ ]:
predicoes_final = pd.concat(predicoes_por_fold, ignore_index=True)
metricas_df = pd.DataFrame(metricas_por_fold)

# Salvar todas as previsões, incluindo respostas inválidas, para auditoria
predicoes_path = f"{OUTPUT_DIR}/predicoes_llama_cv_5_folds_1100.csv"
metricas_path = f"{OUTPUT_DIR}/metricas_llama_cv_5_folds_1100.csv"
resumo_path = f"{OUTPUT_DIR}/resumo_metricas_llama_cv_5_folds_1100.json"
matriz_path = f"{OUTPUT_DIR}/matriz_confusao_llama_cv_5_folds_1100.csv"

predicoes_final.to_csv(predicoes_path, index=False, encoding="utf-8-sig")
metricas_df.to_csv(metricas_path, index=False, encoding="utf-8-sig")

validos = predicoes_final["predicao_llama"].isin(CLASSES)
# Não remover respostas inválidas: elas contam como erros na avaliação.
y_true = predicoes_final["rotulo_final"]
y_pred = predicoes_final["predicao_llama"]

resumo = {
    "modelo": MODEL_ID,
    "base": "base_tcc_final_1100.csv",
    "n_comentarios": int(len(predicoes_final)),
    "n_predicoes_validas": int(validos.sum()),
    "n_predicoes_invalidas": int((~validos).sum()),
    "epocas_por_fold": EPOCHS,
    "folds_executados": [int(x) for x in sorted(metricas_df["fold"].unique())],
    "quantidade_folds_executados": int(metricas_df["fold"].nunique()),
    "macro_f1_media_folds_executados": float(metricas_df["macro_f1"].mean()),
    "macro_f1_dp_folds_executados": float(metricas_df["macro_f1"].std(ddof=1)),
    "precisao_macro_media_folds_executados": float(metricas_df["precisao_macro"].mean()),
    "precisao_macro_dp_folds_executados": float(metricas_df["precisao_macro"].std(ddof=1)),
    "acuracia_media_folds_executados": float(metricas_df["acuracia"].mean()),
    "acuracia_dp_folds_executados": float(metricas_df["acuracia"].std(ddof=1)),
    "nota": "Todas as previsões entram nas métricas; respostas inválidas contam como incorretas."
}
for classe in CLASSES:
    resumo[f"f1_{classe}_media_folds"] = float(metricas_df[f"f1_{classe}"].mean())
    resumo[f"f1_{classe}_dp_folds"] = float(metricas_df[f"f1_{classe}"].std(ddof=1))

with open(resumo_path, "w", encoding="utf-8") as f:
    json.dump(resumo, f, ensure_ascii=False, indent=2)

matriz = confusion_matrix(y_true, y_pred, labels=CLASSES)
pd.DataFrame(
    matriz,
    index=[f"Real: {c}" for c in CLASSES],
    columns=[f"Previsto: {c}" for c in CLASSES]
).to_csv(matriz_path, encoding="utf-8-sig")

print("=== RESUMO ===")
print(json.dumps(resumo, ensure_ascii=False, indent=2))
print("\n=== MÉTRICAS POR FOLD ===")
display(metricas_df)
print("\n=== MATRIZ DE CONFUSÃO (somente previsões válidas) ===")
display(pd.DataFrame(matriz, index=[f"Real: {c}" for c in CLASSES], columns=[f"Previsto: {c}" for c in CLASSES]))
print("\nArquivos salvos em:", OUTPUT_DIR)
print(predicoes_path)
print(metricas_path)
print(resumo_path)
print(matriz_path)

## Etapa 7 — Exportação dos resultados

Os arquivos CSV e JSON são compactados em um único arquivo ZIP e disponibilizados para download no Google Colab.

Recomenda-se guardar o arquivo exportado junto aos registros experimentais, identificando a versão do notebook, os arquivos de entrada e os parâmetros utilizados.

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/resultados_llama_cv_1100", "zip", OUTPUT_DIR)
print("ZIP criado:", zip_path)

from google.colab import files
files.download(zip_path)

## 8. Arquivos produzidos

A pasta `/content/resultados_llama_1100/` reúne os resultados intermediários e consolidados dos cinco folds:

| Arquivo | Descrição |
|---|---|
| `predicoes_fold_N.csv` | Previsões e respostas brutas do fold `N` |
| `metricas_por_fold_parcial.csv` | Métricas dos folds executados |
| `predicoes_llama_cv_5_folds_1100.csv` | Previsões consolidadas |
| `metricas_llama_cv_5_folds_1100.csv` | Métricas por fold |
| `resumo_metricas_llama_cv_5_folds_1100.json` | Resumo dos indicadores |
| `matriz_confusao_llama_cv_5_folds_1100.csv` | Matriz de confusão consolidada |

## 9. Considerações para reprodução

Para reproduzir o experimento, devem ser mantidos os mesmos arquivos de entrada, a mesma associação entre comentários e folds, o identificador do modelo-base, os parâmetros de quantização e LoRA, os parâmetros de treinamento e a semente aleatória.

As versões das bibliotecas e o ambiente de execução também podem influenciar os resultados. Por isso, recomenda-se registrar as versões exibidas na etapa de preparação do ambiente.

**Interpretação:** quando os cinco folds forem executados com sucesso, as métricas por fold e as estatísticas agregadas representam a validação cruzada de cinco partições. A média e o desvio-padrão devem ser calculados a partir dos resultados individuais de cada fold.

## 10. Referências técnicas

- HUGGING FACE. *Transformers documentation*. Disponível em: https://huggingface.co/docs/transformers
- HUGGING FACE. *PEFT documentation*. Disponível em: https://huggingface.co/docs/peft
- HUGGING FACE. *Bitsandbytes integration*. Disponível em: https://huggingface.co/docs/transformers/quantization/bitsandbytes
- META. *Llama 3.1 8B Instruct*. Disponível em: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

As referências acima documentam as ferramentas utilizadas na implementação. As referências acadêmicas sobre LoRA, QLoRA e métricas de classificação também devem constar na bibliografia do TCC, de acordo com as obras efetivamente utilizadas no trabalho.